# 02 — Load, clean, and prepare offline data

Dedicated corpus preparation for the offline sandbox:

| Source | Input | Output |
| --- | --- | --- |
| Fixture catalog | `data/fixtures/manifest.csv` + text files | `data/runtime/prepared/fixtures_prepared.jsonl` |
| HF mini slice | `data/fixtures/hf/docclass_mini.jsonl` | `…/hf_prepared.jsonl` |
| LegalBench | `data/fixtures/legalbench/contract_qa.jsonl` | `…/legalbench_prepared.jsonl` |
| Agent gold | `data/fixtures/agents/*.jsonl` | `…/agents/*_prepared.jsonl` |
| Sorter pilot | class-balanced view of fixtures | `…/sorter_pilot_balanced.jsonl` |

Cleaning is **network-free**: normalize whitespace, drop empty/incomplete rows, parse `expected_fields` JSON, attach file text.

Prerequisite: notebook **01** (or an existing `.env` + activatable profile).

In [ ]:
from __future__ import annotations

import json
import os
import sys
from collections import Counter
from pathlib import Path

ROOT = Path(os.environ.get("SANDBOX_ROOT") or Path.cwd())
if (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT / "src"))
    os.environ.setdefault("SANDBOX_ROOT", str(ROOT))

from mailroom_sandbox.datasets import load_manifest
from mailroom_sandbox.prep import (
    clean_hf_rows,
    clean_legalbench_rows,
    clean_manifest_rows,
    ensure_dotenv_from_example,
    prepare_offline_datasets,
    prepared_dir,
)
from mailroom_sandbox.runtime import activate

ensure_dotenv_from_example()
activate(os.environ.get("SANDBOX_PROFILE", "ollama"))
print("prepared_dir →", prepared_dir())

## Inspect raw catalog

In [ ]:
raw = load_manifest()
print(f"manifest rows: {len(raw)}")
print("classes:", sorted({r["expected_doc_class"] for r in raw}))
print("stages:", Counter(r.get("expected_stage") for r in raw))
raw[0]

## Clean step-by-step (fixtures → HF → LegalBench)

In [ ]:
fixtures, fix_report = clean_manifest_rows()
hf_rows, hf_report = clean_hf_rows()
lb_rows, lb_report = clean_legalbench_rows()

for report in (fix_report, hf_report, lb_report):
    print(json.dumps(report.as_dict(), indent=2))

assert fix_report.kept_rows > 0, "no fixture rows survived cleaning"
assert hf_report.kept_rows > 0, "HF mini slice empty"
assert lb_report.kept_rows > 0, "LegalBench slice empty"

print("\nexample fixture id/class/chars:",
      fixtures[0]["id"], fixtures[0]["expected_doc_class"], fixtures[0]["char_count"])
print(fixtures[0]["text"][:280], "…")

## Write all prepared artifacts

`prepare_offline_datasets()` re-runs cleaners and writes JSONL + `prep_manifest.json`.

In [ ]:
summary = prepare_offline_datasets()
print(json.dumps({
    "prepared_at": summary["prepared_at"],
    "fingerprint": summary["fingerprint"],
    "counts": summary["counts"],
    "artifacts": summary["artifacts"],
    "manifest": summary["manifest"],
}, indent=2))

assert summary["counts"]["fixtures"] >= 8
assert Path(ROOT, summary["manifest"]).is_file()

## Optional pandas overview

In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(fixtures)
    display(df[["id", "expected_doc_class", "expected_stage", "char_count", "filename"]])
    print(df.groupby("expected_doc_class")["char_count"].agg(["count", "mean", "min", "max"]))
except ImportError:
    print("pandas not installed — pip install -e '.[notebooks]' (already in the Docker image)")

## Next

Open **`03_offline_sandbox_smoke.ipynb`** for a mock pilot against the prepared corpus (no live LLM).